# NB10 — Analysis & Paper Assets

Aggregates every upstream `outputs/` artifact into paper-ready **tables, figures, and LaTeX**.
Each section is guarded — if an upstream notebook hasn't run yet it is **skipped cleanly** (never
emits NaN rows or broken figures). Re-run after any upstream notebook to refresh.

Outputs → `outputs/paper/` (CSV + `.tex` per table, PNG per figure).


In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05)
PAPER="../outputs/paper"; os.makedirs(PAPER, exist_ok=True)
def exists(p): return os.path.isfile(p)
def load_json(p): return json.load(open(p)) if exists(p) else None
def load_csv(p):  return pd.read_csv(p) if exists(p) else None
def save_table(df, name):
    df.to_csv(f"{PAPER}/{name}.csv", index=False)
    try:
        with open(f"{PAPER}/{name}.tex","w") as f: f.write(df.to_latex(index=False, escape=True))
    except Exception as e: print("  (latex skip:", e, ")")
    print(f"  ✅ {name}: CSV + tex")
print("paper dir:", PAPER)

In [ ]:
# ── Table 1: Main results (baselines + per-model + ensemble), 20% test ──────
rows=[]
bl=load_csv("../outputs/baselines/baseline_results.csv")
if bl is not None:
    for _,r in bl.iterrows():
        rows.append({"System":r["model"],"Type":"baseline","Macro-F1":r["macro_f1"],
                     "Weighted-F1":r["weighted_f1"],"Accuracy":r["accuracy"],"MCC":r["mcc"],"AUROC":r.get("auroc",np.nan)})
pm=load_csv("../outputs/models_main/per_run_summary.csv")
if pm is not None:
    for mk,g in pm.groupby("model"):
        rows.append({"System":mk,"Type":"transformer (mean±std)",
                     "Macro-F1":f"{g['macro_f1'].mean():.4f}±{g['macro_f1'].std():.4f}",
                     "Weighted-F1":f"{g['weighted_f1'].mean():.4f}±{g['weighted_f1'].std():.4f}",
                     "Accuracy":f"{g['accuracy'].mean():.4f}","MCC":f"{g['mcc'].mean():.4f}","AUROC":f"{g['auroc'].mean():.4f}"})
ens=load_json("../outputs/ensemble/ensemble_test_metrics.json")
if ens is not None:
    rows.append({"System":f"Ensemble ({ens.get('system','-')})","Type":"ours","Macro-F1":ens["macro_f1"],
                 "Weighted-F1":ens["weighted_f1"],"Accuracy":ens["accuracy"],"MCC":ens["mcc"],"AUROC":ens["macro_auroc"]})
if rows:
    t1=pd.DataFrame(rows)
    print("TABLE 1 — Main results (20% test)"); print(t1.to_string(index=False)); save_table(t1,"table1_main_results")
else:
    print("⏭ Table 1 skipped (run NB04/NB05/NB06 first)")

In [ ]:
# ── Table 2 + Fig: Per-class F1 of the ensemble ─────────────────────────────
ens=load_json("../outputs/ensemble/ensemble_test_metrics.json")
if ens is not None and ens.get("per_class_f1"):
    pcf=ens["per_class_f1"]
    t2=pd.DataFrame({"Class":list(pcf.keys()),"F1":list(pcf.values())}).sort_values("F1",ascending=False)
    print("TABLE 2 — Per-class F1 (ensemble, 20% test)"); print(t2.to_string(index=False)); save_table(t2,"table2_per_class")
    fig,ax=plt.subplots(figsize=(8,4))
    sns.barplot(data=t2,y="Class",x="F1",ax=ax,palette="viridis")
    for i,v in enumerate(t2["F1"]): ax.text(v+0.01,i,f"{v:.3f}",va="center",fontsize=9)
    ax.set_xlim(0,1); ax.set_title("Per-Class F1 — Ensemble (20% test)",fontweight="bold")
    plt.tight_layout(); plt.savefig(f"{PAPER}/fig_per_class_f1.png",dpi=150,bbox_inches="tight"); plt.show()
else:
    print("⏭ Table 2 skipped (run NB06 first)")

In [ ]:
# ── Table 3 + Fig: Component ablation ───────────────────────────────────────
ca=load_csv("../outputs/ablation/component_ablation.csv")
if ca is not None:
    t3=ca[["config","macro_f1","weighted_f1","accuracy","mcc"]].copy()
    print("TABLE 3 — Component ablation"); print(t3.to_string(index=False)); save_table(t3,"table3_component_ablation")
    fig,ax=plt.subplots(figsize=(9,4))
    ax.plot(t3["config"],t3["macro_f1"],marker="o",lw=2,label="Macro-F1")
    ax.plot(t3["config"],t3["weighted_f1"],marker="s",lw=2,label="Weighted-F1")
    for x,y in zip(range(len(t3)),t3["macro_f1"]): ax.text(x,y+0.005,f"{y:.3f}",ha="center",fontsize=8)
    ax.set_title("Additive Component Ablation",fontweight="bold"); ax.set_ylabel("F1"); ax.legend()
    plt.xticks(rotation=30,ha="right"); plt.tight_layout()
    plt.savefig(f"{PAPER}/fig_ablation.png",dpi=150,bbox_inches="tight"); plt.show()
else:
    print("⏭ Table 3 skipped (run NB08 first)")

In [ ]:
# ── Table 4: Taxonomy ablation (5 vs 9 class) ───────────────────────────────
ta=load_csv("../outputs/ablation/taxonomy_ablation.csv")
if ta is not None:
    t4=ta[["config","macro_f1","weighted_f1","accuracy","mcc"]].copy()
    print("TABLE 4 — Taxonomy ablation (5 vs 9 class)"); print(t4.to_string(index=False)); save_table(t4,"table4_taxonomy")
else:
    print("⏭ Table 4 skipped (run NB08 first)")

In [ ]:
# ── Table 5 + Fig: Robustness (held-outs) ───────────────────────────────────
rb=load_csv("../outputs/robustness/robustness_summary.csv")
if rb is not None:
    t5=rb[["config","held_out","n_test","macro_f1","weighted_f1","accuracy","mcc"]].copy()
    print("TABLE 5 — Robustness (source/script held-out)"); print(t5.to_string(index=False)); save_table(t5,"table5_robustness")
    fig,ax=plt.subplots(figsize=(9,4))
    sns.barplot(data=t5,x="held_out",y="macro_f1",ax=ax,palette="rocket")
    for i,v in enumerate(t5["macro_f1"]): ax.text(i,v+0.005,f"{v:.3f}",ha="center",fontsize=9)
    ax.set_title("Cross-domain Robustness (Macro-F1 on unseen held-out)",fontweight="bold"); ax.set_ylim(0,1)
    plt.xticks(rotation=20,ha="right"); plt.tight_layout()
    plt.savefig(f"{PAPER}/fig_robustness.png",dpi=150,bbox_inches="tight"); plt.show()
else:
    print("⏭ Table 5 skipped (run NB07 first)")

In [ ]:
# ── Table 6 + Fig: Base-paper head-to-head ──────────────────────────────────
cmp=load_json("../outputs/basepaper/comparison.json")
if cmp is not None:
    ours,base=cmp["ours"],cmp["base_paper"]
    rows=[{"Class":c,"Ours":ours["per_class_f1"][c],"Base paper":base["per_class"].get(c,np.nan)} for c in ours["per_class_f1"]]
    rows.append({"Class":"MACRO-F1","Ours":ours["macro_f1"],"Base paper":base["macro_f1"]})
    t6=pd.DataFrame(rows)
    print("TABLE 6 — vs Base paper (facebook 15% test)"); print(t6.to_string(index=False)); save_table(t6,"table6_basepaper")
    pc=t6[t6["Class"]!="MACRO-F1"]
    x=np.arange(len(pc)); w=0.38; fig,ax=plt.subplots(figsize=(9,4))
    ax.bar(x-w/2,pc["Ours"],w,label="Ours"); ax.bar(x+w/2,pc["Base paper"],w,label="Base paper")
    ax.set_xticks(x); ax.set_xticklabels(pc["Class"],rotation=20,ha="right"); ax.set_ylim(0,1)
    ax.set_title("Per-class F1: Ours vs Base paper",fontweight="bold"); ax.legend()
    plt.tight_layout(); plt.savefig(f"{PAPER}/fig_basepaper.png",dpi=150,bbox_inches="tight"); plt.show()
else:
    print("⏭ Table 6 skipped (run NB09 first)")

In [ ]:
# ── Manifest of generated assets ────────────────────────────────────────────
assets=sorted(os.listdir(PAPER))
print(f"\n📦 {len(assets)} assets in {PAPER}:")
for a in assets: print("   ", a)